# Sentiment model bake-off

Compare several Hugging Face sentiment models on IPS discovery notes.

**Goal:** pick the model that best separates pain (negative) vs likes/praise (positive) on short workplace phrases.

Models:
1. `Thi144/sentiment-distilbert-7class` — current pipeline (7-class DistilBERT)
2. `distilbert-base-uncased-finetuned-sst-2-english` — classic DistilBERT SST-2
3. `cardiffnlp/twitter-roberta-base-sentiment-latest` — RoBERTa Twitter 3-class
4. `cardiffnlp/twitter-roberta-base-sentiment` — RoBERTa Twitter 3-class (earlier)

Evaluation: hand-labeled gold set (domain phrases) + agreement/distribution on real challenges.

## Results (already run)

Gold-set ranking (30 hand-labeled IPS phrases):

| Rank | Model | Accuracy | Macro recall | + share on challenges sample |
|------|-------|----------|--------------|------------------------------|
| 1 | `distilbert-base-uncased-finetuned-sst-2-english` | **0.767** | 0.622 | 13% |
| 2 | `cardiffnlp/twitter-roberta-base-sentiment` | 0.633 | **0.733** | 3% |
| 3 | `cardiffnlp/twitter-roberta-base-sentiment-latest` | 0.600 | 0.689 | 2% |
| 4 | `Thi144/sentiment-distilbert-7class` (current) | 0.600 | 0.511 | **47%** |

**Takeaway:** SST-2 DistilBERT wins overall accuracy and catches pain best.  
Cardiff Twitter RoBERTa is more balanced (best macro recall) and rarely false-positives on challenges.  
Current 7-class model over-labels challenges as positive (~half the sample) — matches the earlier quality issue.

Artifacts: `output/processed/sentiment_model_leaderboard.csv` (+ gold/real sample CSVs).


In [1]:
%pip install -q -r ../requirements.txt
%pip install -q "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

ROOT = Path("..").resolve() if (Path("..") / "setup").is_dir() else Path(".").resolve()
os.chdir(ROOT)

# Load HF token if present (same as setup/sentiment_analysis.py)
env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key == "HF_TOKEN" and val and "HF_TOKEN" not in os.environ:
            os.environ["HF_TOKEN"] = val

PROCESSED = ROOT / "output" / "processed"
OUT_DIR = PROCESSED
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    {
        "id": "distilbert-7class",
        "name": "Thi144/sentiment-distilbert-7class",
        "kind": "7class",
    },
    {
        "id": "distilbert-sst2",
        "name": "distilbert-base-uncased-finetuned-sst-2-english",
        "kind": "sst2",
    },
    {
        "id": "roberta-twitter-latest",
        "name": "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "kind": "cardiff",
    },
    {
        "id": "roberta-twitter",
        "name": "cardiffnlp/twitter-roberta-base-sentiment",
        "kind": "cardiff",
    },
]

print("ROOT:", ROOT)
print("Models:", [m["id"] for m in MODELS])

ROOT: /Users/lilscott/Code/IPS/simple/notebook
Models: ['distilbert-7class', 'distilbert-sst2', 'roberta-twitter-latest', 'roberta-twitter']


## 1. Gold set (domain phrases)

Clear workplace discovery lines with expected polarity. Used for accuracy ranking.

In [3]:
GOLD = pd.DataFrame(
    [
        # negative — clear pain
        ("Repeating data entry", "negative"),
        ("No full access to IPS", "negative"),
        ("Initial user frustration (learning curve)", "negative"),
        ("System crashes when uploading large plans", "negative"),
        ("Too many clicks to complete a simple action", "negative"),
        ("Missing information between departments", "negative"),
        ("Slow performance and freezes during inspections", "negative"),
        ("Can't find property history easily", "negative"),
        ("Manual workaround in Excel every week", "negative"),
        ("Duplicate entry across multiple systems", "negative"),
        ("Navigating between multiple screens", "negative"),
        ("Looking for details or specific information", "negative"),
        ("Fills up quickly with things like electrical inspections", "negative"),
        ("Hard to track referral status", "negative"),
        ("Internet drops and we lose what we entered", "negative"),
        # positive — likes / praise
        ("Likes ability to filter in Camino", "positive"),
        ("Organizing by tax number is useful", "positive"),
        ("Likes that everything is in Camino – database for that", "positive"),
        ("IPS does well: summary search works", "positive"),
        ("Likes how detailed the info we can input is", "positive"),
        ("Likes notes tab – nice that it's separate from action log", "positive"),
        ("Many ways to search – address, owner name works well", "positive"),
        ("Documents and images are saved well in the system", "positive"),
        ("He likes IPS for what he uses it for", "positive"),
        ("Easy to see what's open and what's closed when searching", "positive"),
        # neutral / descriptive capability (not praise, not pain)
        ("Ability to contest, schedule by HO", "neutral"),
        ("Ability to issue decision by ALJ", "neutral"),
        ("Ability to cross-reference account other than address", "neutral"),
        ("Ability to search by address", "neutral"),
        ("Dashboard for supervisors", "neutral"),
    ],
    columns=["text", "gold"],
)
print(GOLD["gold"].value_counts().to_string())
display(GOLD)

gold
negative    15
positive    10
neutral      5


,text,gold
0,Repeating data entry,negative
1,No full access to IPS,negative
2,Initial user frustration (learning curve),negative
3,System crashes when uploading large plans,negative
4,Too many clicks to complete a simple action,negative
5,Missing information between departments,negative
6,Slow performance and freezes during inspections,negative
7,Can't find property history easily,negative
8,Manual workaround in Excel every week,negative
9,Duplicate entry across multiple systems,negative


## 2. Model helpers

Normalize each model's raw labels → `negative` / `neutral` / `positive`.

In [4]:
SEVEN_CLASS = {
    0: "negative",
    1: "negative",
    2: "negative",
    3: "neutral",
    4: "positive",
    5: "positive",
    6: "positive",
    "Very Negative": "negative",
    "Negative": "negative",
    "Slightly Negative": "negative",
    "Neutral": "neutral",
    "Slightly Positive": "positive",
    "Positive": "positive",
    "Very Positive": "positive",
}

CARDIFF = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive",
    "negative": "negative",
    "neutral": "neutral",
    "positive": "positive",
}

SST2 = {
    "LABEL_0": "negative",
    "LABEL_1": "positive",
    "NEGATIVE": "negative",
    "POSITIVE": "positive",
    "negative": "negative",
    "positive": "positive",
}


def normalize_label(raw, kind: str, score: float | None = None) -> str:
    raw_s = str(raw)
    if kind == "7class":
        # Model returns LABEL_0..LABEL_6 (or names / ints)
        key = raw_s
        if key.startswith("LABEL_"):
            key = key.replace("LABEL_", "", 1)
        if key.isdigit():
            return SEVEN_CLASS.get(int(key), "neutral")
        return SEVEN_CLASS.get(raw_s, SEVEN_CLASS.get(key, "neutral"))
    if kind == "cardiff":
        return CARDIFF.get(raw_s, CARDIFF.get(raw_s.lower(), "neutral"))
    if kind == "sst2":
        # Binary model: low-confidence → neutral so it can compete fairly
        polarity = SST2.get(raw_s, SST2.get(raw_s.upper(), SST2.get(raw_s.lower(), "neutral")))
        if score is not None and score < 0.65:
            return "neutral"
        return polarity
    return "neutral"


def build_pipe(model_name: str, kind: str):
    """Build a HF sentiment pipeline; Cardiff needs explicit tokenizer/model."""
    if kind == "cardiff":
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
        return pipeline(
            "sentiment-analysis",
            model=mdl,
            tokenizer=tok,
            truncation=True,
            max_length=512,
            top_k=None,
        )
    return pipeline(
        "sentiment-analysis",
        model=model_name,
        truncation=True,
        max_length=512,
        top_k=None,
    )


def score_texts(clf, texts: list[str], kind: str) -> list[tuple[str, float]]:
    out: list[tuple[str, float]] = []
    for text in texts:
        result = clf(text)
        if result and isinstance(result[0], list):
            ranked = sorted(result[0], key=lambda r: r["score"], reverse=True)
        else:
            ranked = result if isinstance(result, list) else [result]
        top = ranked[0]
        label = normalize_label(top["label"], kind, float(top["score"]))
        out.append((label, float(top["score"])))
    return out


def metrics(pred: pd.Series, gold: pd.Series) -> dict:
    correct = (pred == gold).sum()
    n = len(gold)
    acc = correct / n if n else 0.0
    by_class = {}
    for c in ["negative", "neutral", "positive"]:
        mask = gold == c
        if mask.any():
            by_class[c] = float((pred[mask] == c).mean())
        else:
            by_class[c] = float("nan")
    # Macro F1-ish recall average (per-class accuracy on gold support)
    recalls = [by_class[c] for c in ["negative", "neutral", "positive"] if gold.eq(c).any()]
    macro = sum(recalls) / len(recalls) if recalls else 0.0
    return {
        "accuracy": acc,
        "macro_recall": macro,
        "neg_recall": by_class["negative"],
        "neu_recall": by_class["neutral"],
        "pos_recall": by_class["positive"],
        "n": n,
    }

print("Helpers ready")


Helpers ready


## 3. Score gold set

First download may take a few minutes.

In [5]:
gold_preds = GOLD[["text", "gold"]].copy()
rows = []
pipes = {}

for m in MODELS:
    mid, name, kind = m["id"], m["name"], m["kind"]
    print(f"\n=== Loading {mid} ({name}) ===")
    t0 = time.perf_counter()
    clf = build_pipe(name, kind)
    load_s = time.perf_counter() - t0
    pipes[mid] = (clf, kind)

    t1 = time.perf_counter()
    scored = score_texts(clf, GOLD["text"].tolist(), kind)
    infer_s = time.perf_counter() - t1

    labels = [s[0] for s in scored]
    scores = [s[1] for s in scored]
    gold_preds[mid] = labels
    gold_preds[f"{mid}_score"] = scores

    mtr = metrics(pd.Series(labels), GOLD["gold"])
    rows.append(
        {
            "model_id": mid,
            "model": name,
            "load_sec": round(load_s, 1),
            "infer_sec": round(infer_s, 2),
            **{k: round(v, 3) if isinstance(v, float) else v for k, v in mtr.items()},
        }
    )
    print(
        f"  accuracy={mtr['accuracy']:.3f}  macro_recall={mtr['macro_recall']:.3f}  "
        f"load={load_s:.1f}s  infer={infer_s:.2f}s"
    )

leaderboard = pd.DataFrame(rows).sort_values(
    ["accuracy", "macro_recall"], ascending=False
).reset_index(drop=True)
display(leaderboard)
display(gold_preds)


=== Loading distilbert-7class (Thi144/sentiment-distilbert-7class) ===


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  accuracy=0.600  macro_recall=0.511  load=1.3s  infer=1.73s

=== Loading distilbert-sst2 (distilbert-base-uncased-finetuned-sst-2-english) ===


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  accuracy=0.767  macro_recall=0.622  load=1.6s  infer=0.35s

=== Loading roberta-twitter-latest (cardiffnlp/twitter-roberta-base-sentiment-latest) ===


...

  accuracy=0.600  macro_recall=0.689  load=2.1s  infer=0.69s

=== Loading roberta-twitter (cardiffnlp/twitter-roberta-base-sentiment) ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  accuracy=0.633  macro_recall=0.733  load=2.3s  infer=0.58s


,model_id,model,load_sec,infer_sec,accuracy,macro_recall,neg_recall,neu_recall,pos_recall,n
0,distilbert-sst2,distilbert-base-uncased-finetuned-sst-2-english,1.6,0.35,0.767,0.622,0.867,0.0,1.0,30
1,roberta-twitter,cardiffnlp/twitter-roberta-base-sentiment,2.3,0.58,0.633,0.733,0.400,1.0,0.8,30
2,roberta-twitter-latest,cardiffnlp/twitter-roberta-base-sentiment-latest,2.1,0.69,0.600,0.689,0.467,1.0,0.6,30
3,distilbert-7class,Thi144/sentiment-distilbert-7class,1.3,1.73,0.600,0.511,0.533,0.0,1.0,30


,text,gold,distilbert-7class,distilbert-7class_score,distilbert-sst2,distilbert-sst2_score,roberta-twitter-latest,roberta-twitter-latest_score,roberta-twitter,roberta-twitter_score
0,Repeating data entry,negative,positive,0.298034,negative,0.985062,neutral,0.832758,neutral,0.764253
1,No full access to IPS,negative,negative,0.359620,negative,0.999743,neutral,0.605116,neutral,0.584593
2,Initial user frustration (learning curve),negative,positive,0.374731,negative,0.998602,negative,0.646541,negative,0.717865
3,System crashes when uploading large plans,negative,negative,0.297464,negative,0.999743,negative,0.780898,negative,0.861971
4,Too many clicks to complete a simple action,negative,negative,0.504309,negative,0.999397,negative,0.680337,negative,0.577816
5,Missing information between departments,negative,negative,0.312651,negative,0.999770,neutral,0.581549,neutral,0.629298
6,Slow performance and freezes during inspections,negative,negative,0.302602,negative,0.999573,negative,0.815475,negative,0.760261
7,Can't find property history easily,negative,negative,0.381408,negative,0.989846,negative,0.678219,negative,0.628366
8,Manual workaround in Excel every week,negative,positive,0.503518,positive,0.992795,neutral,0.835303,neutral,0.880866
9,Duplicate entry across multiple systems,negative,negative,0.291086,negative,0.999579,neutral,0.787707,neutral,0.808487


In [6]:
fig = px.bar(
    leaderboard,
    x="model_id",
    y=["accuracy", "macro_recall", "neg_recall", "neu_recall", "pos_recall"],
    barmode="group",
    title="Gold-set performance by model",
    labels={"value": "score", "variable": "metric"},
)
fig.update_layout(yaxis_range=[0, 1.05], template="plotly_white")
fig.show()

winner = leaderboard.iloc[0]
print(
    f"Winner on gold set: {winner['model_id']} "
    f"({winner['model']}) — accuracy={winner['accuracy']:.3f}, "
    f"macro_recall={winner['macro_recall']:.3f}"
)

Winner on gold set: distilbert-sst2 (distilbert-base-uncased-finetuned-sst-2-english) — accuracy=0.767, macro_recall=0.622


### Gold-set errors (per model)

Where each model disagrees with the hand labels.

In [7]:
for mid in [m["id"] for m in MODELS]:
    bad = gold_preds.loc[gold_preds[mid] != gold_preds["gold"], ["text", "gold", mid, f"{mid}_score"]]
    print(f"\n--- {mid}: {len(bad)} / {len(gold_preds)} errors ---")
    display(bad)


--- distilbert-7class: 12 / 30 errors ---


,text,gold,distilbert-7class,distilbert-7class_score
0,Repeating data entry,negative,positive,0.298034
2,Initial user frustration (learning curve),negative,positive,0.374731
8,Manual workaround in Excel every week,negative,positive,0.503518
10,Navigating between multiple screens,negative,positive,0.498447
11,Looking for details or specific information,negative,positive,0.501231
12,Fills up quickly with things like electrical i...,negative,positive,0.544013
13,Hard to track referral status,negative,positive,0.385362
25,"Ability to contest, schedule by HO",neutral,positive,0.530095
26,Ability to issue decision by ALJ,neutral,positive,0.539664
27,Ability to cross-reference account other than ...,neutral,positive,0.516378



--- distilbert-sst2: 7 / 30 errors ---


,text,gold,distilbert-sst2,distilbert-sst2_score
8,Manual workaround in Excel every week,negative,positive,0.992795
10,Navigating between multiple screens,negative,positive,0.990106
25,"Ability to contest, schedule by HO",neutral,positive,0.999391
26,Ability to issue decision by ALJ,neutral,positive,0.999576
27,Ability to cross-reference account other than ...,neutral,positive,0.995006
28,Ability to search by address,neutral,positive,0.999446
29,Dashboard for supervisors,neutral,negative,0.743598



--- roberta-twitter-latest: 12 / 30 errors ---


,text,gold,roberta-twitter-latest,roberta-twitter-latest_score
0,Repeating data entry,negative,neutral,0.832758
1,No full access to IPS,negative,neutral,0.605116
5,Missing information between departments,negative,neutral,0.581549
8,Manual workaround in Excel every week,negative,neutral,0.835303
9,Duplicate entry across multiple systems,negative,neutral,0.787707
10,Navigating between multiple screens,negative,neutral,0.820628
11,Looking for details or specific information,negative,neutral,0.901827
12,Fills up quickly with things like electrical i...,negative,neutral,0.877758
15,Likes ability to filter in Camino,positive,neutral,0.702929
16,Organizing by tax number is useful,positive,neutral,0.575248



--- roberta-twitter: 11 / 30 errors ---


,text,gold,roberta-twitter,roberta-twitter_score
0,Repeating data entry,negative,neutral,0.764253
1,No full access to IPS,negative,neutral,0.584593
5,Missing information between departments,negative,neutral,0.629298
8,Manual workaround in Excel every week,negative,neutral,0.880866
9,Duplicate entry across multiple systems,negative,neutral,0.808487
10,Navigating between multiple screens,negative,neutral,0.782426
11,Looking for details or specific information,negative,neutral,0.902055
12,Fills up quickly with things like electrical i...,negative,neutral,0.709066
13,Hard to track referral status,negative,neutral,0.647349
23,He likes IPS for what he uses it for,positive,neutral,0.574195


## 4. Real challenges sample

Score a stratified sample of real pain-point lines (no gold labels). Compare label mix and pairwise agreement with the current DistilBERT-7class baseline.

In [8]:
challenges = pd.read_csv(PROCESSED / "challenges.csv")
text_col = "pain_points"

# Prefer short + medium lines (where models struggle most); cap for speed
sample_n = 120
work = challenges[["department", "focus_group", "source", text_col]].dropna().copy()
work["n_words"] = work[text_col].astype(str).str.split().str.len()
work = work.loc[work["n_words"].between(3, 40)]

sample = (
    work.groupby("source", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), sample_n // 2), random_state=42))
    .reset_index(drop=True)
)
if len(sample) < sample_n:
    extra = work.loc[~work.index.isin(sample.index)].sample(
        min(sample_n - len(sample), len(work)), random_state=42
    )
    sample = pd.concat([sample, extra], ignore_index=True)

print(f"Sample size: {len(sample)} (of {len(work)} eligible rows)")
print(sample["source"].value_counts().to_string())

real_preds = sample.copy()
for mid, (clf, kind) in pipes.items():
    print(f"Scoring real sample with {mid}…")
    scored = score_texts(clf, sample[text_col].astype(str).tolist(), kind)
    real_preds[mid] = [s[0] for s in scored]
    real_preds[f"{mid}_score"] = [s[1] for s in scored]

mix = (
    pd.DataFrame({mid: real_preds[mid].value_counts(normalize=True) for mid in pipes})
    .fillna(0)
    .reindex(["negative", "neutral", "positive"])
)
print("\nLabel mix on real sample:")
display(mix.round(3))

fig2 = px.bar(
    mix.reset_index().melt(id_vars="index", var_name="model", value_name="share"),
    x="model",
    y="share",
    color="index",
    barmode="stack",
    title="Sentiment mix on real challenges sample",
    labels={"index": "sentiment", "share": "share of sample"},
)
fig2.update_layout(yaxis_tickformat=".0%", template="plotly_white")
fig2.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lilscott/Code/IPS/simple/notebook/output/processed/challenges.csv'

In [ ]:
ids = list(pipes.keys())
agree_rows = []
for i, a in enumerate(ids):
    for b in ids[i + 1 :]:
        agree = float((real_preds[a] == real_preds[b]).mean())
        agree_rows.append({"a": a, "b": b, "agreement": round(agree, 3)})
agree_df = pd.DataFrame(agree_rows).sort_values("agreement", ascending=False)
print("Pairwise agreement on real sample:")
display(agree_df)

# Spotlight cases where current 7class says positive but others disagree (known pain point)
baseline = "distilbert-7class"
others = [i for i in ids if i != baseline]
flag = real_preds[baseline] == "positive"
for o in others:
    flag = flag & (real_preds[o] != "positive")
suspect = real_preds.loc[
    flag, [text_col, "source", baseline] + others
].head(25)
print(
    f"\nCases where {baseline}=positive but others disagree: {flag.sum()} "
    f"(showing up to 25)"
)
display(suspect)

## 5. Verdict + save

Primary ranking = gold-set **accuracy**, then **macro recall**.  
Secondary signal = fewer false “positive” on short pain phrases in the real sample.

In [ ]:
# False-positive rate proxy on real sample: share labeled positive
# (challenges are mostly pain — lower positive share is usually healthier)
pos_share = (
    real_preds[[m["id"] for m in MODELS]]
    .apply(lambda s: (s == "positive").mean())
    .rename("positive_share_on_challenges")
)

summary = leaderboard.merge(
    pos_share.rename_axis("model_id").reset_index(),
    on="model_id",
    how="left",
)
summary["positive_share_on_challenges"] = summary["positive_share_on_challenges"].round(3)
summary = summary.sort_values(
    ["accuracy", "macro_recall"], ascending=False
).reset_index(drop=True)

best = summary.iloc[0]
print("=== Leaderboard ===")
display(summary)
print(
    f"\nRecommended model: {best['model']}\n"
    f"  gold accuracy={best['accuracy']:.3f}, macro_recall={best['macro_recall']:.3f}\n"
    f"  positive share on challenges sample={best['positive_share_on_challenges']:.1%}\n"
    f"\nTo adopt it, set SENTIMENT_MODEL in setup/sentiment_analysis.py and "
    f"adjust label normalization for that model's label space."
)

summary.to_csv(OUT_DIR / "sentiment_model_leaderboard.csv", index=False, encoding="utf-8-sig")
gold_preds.to_csv(OUT_DIR / "sentiment_model_gold_preds.csv", index=False, encoding="utf-8-sig")
real_preds.to_csv(OUT_DIR / "sentiment_model_real_sample.csv", index=False, encoding="utf-8-sig")
print(f"\nSaved → {OUT_DIR / 'sentiment_model_leaderboard.csv'}")